# MathForge — overnight QLoRA fine-tune

Trains a LoRA adapter on **Qwen2.5-1.5B-Instruct** (4-bit) to generate elegant competition-math problems, using `data/train.jsonl` (prompts conditioned on topic / difficulty / elegance).

**Runtime:** set `Runtime → Change runtime type → GPU` (free **T4** is enough for 1.5B; **L4/A100** on Colab Pro for 3B).

**Overnight-safe:** checkpoints to Google Drive every 50 steps and auto-resumes if Colab disconnects. Run the cells top to bottom, then leave it.

The repo is private, so this notebook is self-contained — you only upload `train.jsonl`.

## 1. Confirm you have a GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies (~2 min)

In [ ]:
!pip install -q -U "transformers>=4.44" "trl>=0.9" "peft>=0.12" "datasets>=2.20" bitsandbytes accelerate

## 3. Mount Google Drive (so checkpoints survive a disconnect)
The adapter + checkpoints are written to `MyDrive/mathforge-qlora/`. If you skip this, they live on the ephemeral VM and vanish when it resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/mathforge-qlora'

## 4. Upload `data/train.jsonl`
Grab it from your machine at `~/Desktop/SLM/data/train.jsonl` (or download **Raw** from the GitHub repo).

In [ ]:
from google.colab import files
up = files.upload()            # pick train.jsonl
DATA = list(up.keys())[0]
print('using', DATA)

## 5. Config
Defaults train on the **elegant** subset (problem-elegance ≥ 3.5, ~2,200 examples). Set `ANSWER_TYPE='integer'` for AIME-style only; raise `MIN_PE` to 4.0 for a smaller, higher-quality run.

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'   # 'Qwen/Qwen2.5-3B-Instruct' on L4/A100
MIN_PE      = 3.5      # min problem-elegance
ANSWER_TYPE = ''       # '' = any; 'integer' = AIME-style answers only
EPOCHS      = 3
MAX_SEQ_LEN = 2048
LR          = 2e-4
SAVE_STEPS  = 50

## 6. Train (this is the long cell — leave it running overnight)

In [ ]:
import os, torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map='auto', torch_dtype=torch.bfloat16)
model.config.use_cache = False
model.gradient_checkpointing_enable()

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                  task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj',
                                  'gate_proj','up_proj','down_proj'])

ds = load_dataset('json', data_files=DATA, split='train')
n_all = len(ds)
if MIN_PE > 0:
    ds = ds.filter(lambda r: (r['meta'].get('problem_elegance') or 0) >= MIN_PE)
if ANSWER_TYPE:
    ds = ds.filter(lambda r: r['meta'].get('answer_type') == ANSWER_TYPE)
ds = ds.map(lambda r: {'text': tok.apply_chat_template(r['messages'], tokenize=False)},
            remove_columns=ds.column_names)
split = ds.train_test_split(test_size=0.03, seed=42)
print(f'{n_all} total -> {len(ds)} after filters -> train {len(split["train"])} / eval {len(split["test"])}')

cfg = SFTConfig(output_dir=OUT, num_train_epochs=EPOCHS,
                per_device_train_batch_size=1, gradient_accumulation_steps=8,
                learning_rate=LR, warmup_ratio=0.05, lr_scheduler_type='cosine',
                logging_steps=5, eval_strategy='steps', eval_steps=SAVE_STEPS,
                save_strategy='steps', save_steps=SAVE_STEPS, save_total_limit=3,
                load_best_model_at_end=True, metric_for_best_model='eval_loss',
                greater_is_better=False, bf16=True, max_length=MAX_SEQ_LEN,
                packing=False, dataset_text_field='text', gradient_checkpointing=True,
                gradient_checkpointing_kwargs={'use_reentrant': False},
                seed=42, report_to='none')
trainer = SFTTrainer(model=model, args=cfg, train_dataset=split['train'],
                     eval_dataset=split['test'], peft_config=lora)

resume = get_last_checkpoint(OUT) if os.path.isdir(OUT) else None
if resume:
    print('resuming from', resume)
trainer.train(resume_from_checkpoint=resume)
trainer.save_model(OUT); tok.save_pretrained(OUT)
print('saved LoRA adapter to', OUT)

## 7. Smoke test — generate a fresh problem

In [ ]:
model.config.use_cache = True
prompt = ('Compose one original, elegant AIME/olympiad-level competition math problem '
          'with a single integer answer in [0, 999]. Topic: Number Theory. '
          'Target difficulty (AoPS 1-10): 6.5. Target problem-elegance (0-5): 4.5. '
          'Give the statement, then a clean solution ending in the integer answer.')
inputs = tok.apply_chat_template([{'role':'user','content':prompt}],
                                 add_generation_prompt=True,
                                 return_tensors='pt', return_dict=True).to(model.device)
out = model.generate(**inputs, max_new_tokens=600, do_sample=True, temperature=0.9, top_p=0.95)
print(tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

The adapter is in `MyDrive/mathforge-qlora/`. To reuse it later, load the base model the same way and `PeftModel.from_pretrained(model, OUT)`.